|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Writing the kernel<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: coalesce the block-table gather<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root, wherever this notebook was opened from
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
import cudalib

This is PagedAttention's gather, with everything else taken away.

A table of rows in memory, an index saying which row each output needs, and a
sum over each gathered row. No softmax, no heads, no block size. Stage 07
builds the real one; this is the part of it that decides the speed.

In [ ]:
### run this cell: the data, and the oracle

N_ROWS, ROW_LEN = 200_000, 128

table = torch.randn(N_ROWS, ROW_LEN, device='cuda')
index = torch.randperm(N_ROWS, device='cuda').to(torch.int32)  # a block table
out   = torch.empty(N_ROWS, device='cuda')

oracle = table[index.long()].sum(dim=1)
useful_bytes = N_ROWS * ROW_LEN * 4
print(f'{useful_bytes/1e6:.0f} MB of rows to gather')

# Exercise 1: the obvious mapping, and what it costs

Give each output row one thread and let it walk its row. Write the kernel,
check it against the oracle, and measure what fraction of the card's
bandwidth you got.

In [ ]:
NAIVE = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One thread per row. It walks the whole row on its own.
__global__ void gather_naive(const float* __restrict__ table,
                             const int* __restrict__ index,
                             float* __restrict__ out,
                             const int n_rows, const int row_len) {

  // which row is this thread responsible for?
  const int r =
  if (r >= n_rows) return;

  // the block table says where that row actually lives
  const float* src =

  // walk it
  float acc = 0.f;


  out[r] = acc;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor out) {
  const int n_rows = index.numel(), row_len = table.size(1);
  const int threads = 256;
  gather_naive<<<   , threads, 0, at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), out.data_ptr<float>(),
      n_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

naive = cudalib.build_source('cc_gather_naive', NAIVE)

In [ ]:
naive.gather(table, index, out)
print('correct:', torch.allclose(out, oracle, rtol=1e-3, atol=1e-2))

peak = cudalib.peak_bandwidth(fresh=True)
ms_naive = 
gb_naive = 

print(f'thread per row: {ms_naive:.3f} ms  {gb_naive:.0f} GB/s  {100*gb_naive/peak:.0f}% of peak')

# Exercise 2: predict the fix before you write it

Do not measure first. Work out, from the sector arithmetic, how much faster a
coalesced version could be.

You will not get a single number, and that is the interesting part. Bound it
from both sides instead: the worst the naive mapping can do, and the best.
Where the measurement lands between them is a fact about the cache, and you
only get to learn it if you wrote the bounds down first.

In [ ]:
floats_per_sector = 

# At any ONE instruction, how many sectors does the warp touch, and how
# much of each one does it use? That is the worst case.
worst_case = 

# Now think about the NEXT instruction. The thread reads src[d+1]. Where
# is that byte? Did you already pay for it? That is the best case.
best_case = 

print(f'stride between neighbouring threads: {ROW_LEN} floats')
print(f'speedup if nothing is cached:        {worst_case}x')
print(f'speedup if the cache catches it all: {best_case}x')

# Exercise 3: one warp per row

Same arithmetic, same answer. Change only which thread touches which byte:
the 32 lanes of a warp walk one row together, then combine their partial sums
with a shuffle.

In [ ]:
COALESCED = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One WARP per row. The 32 lanes walk the row together, so at every step they
// read 32 consecutive floats: one transaction instead of 32.
__global__ void gather_coalesced(const float* __restrict__ table,
                                 const int* __restrict__ index,
                                 float* __restrict__ out,
                                 const int n_rows, const int row_len) {

  // careful: the thread index now picks a LANE, not a row
  const int warp =
  const int lane =
  if (warp >= n_rows) return;

  const float* src = table + (long)index[warp] * row_len;

  // lane L takes elements L, L+32, L+64, ... so neighbouring lanes are
  // always on neighbouring addresses
  float acc = 0.f;


  // the row's total is now spread across 32 registers. Butterfly them
  // together with __shfl_xor_sync. Every lane in the mask must reach it.


  if (lane == 0) out[warp] = acc;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor out) {
  const int n_rows = index.numel(), row_len = table.size(1);
  const int threads = 256, warps_per_block = threads / 32;

  // how many blocks now? each block only covers warps_per_block rows.
  gather_coalesced<<<   , threads, 0,
                     at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), out.data_ptr<float>(),
      n_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

fast = cudalib.build_source('cc_gather_fast', COALESCED)

In [ ]:
fast.gather(table, index, out)
print('correct:', torch.allclose(out, oracle, rtol=1e-3, atol=1e-2))

ms_fast = 
gb_fast = 

print(f'thread per row: {ms_naive:.3f} ms  {gb_naive:6.0f} GB/s  {100*gb_naive/peak:3.0f}% of peak')
print(f'warp per row:   {ms_fast:.3f} ms  {gb_fast:6.0f} GB/s  {100*gb_fast/peak:3.0f}% of peak')
print(f'\nmeasured speedup:  {ms_naive/ms_fast:.2f}x')
print(f'predicted range:   {best_case}x .. {worst_case}x')

### Before you move on

Three things in your output deserve a sentence each, written down before you
open the solution:

1. Where inside your Exercise 2 bounds did the measurement land? Nearer the
   worst case or the best?
2. A thread in the naive kernel reads `src[d]`, then `src[d+1]`. Did it pay
   for the second one? So what exactly did the naive mapping waste, if not
   bytes?
3. The coalesced version probably reports more than 100% of peak bandwidth.
   That is not a bug in the measurement. What is `useful_bytes` counting, and
   what is the hardware actually moving?

Then go and do this to the real kernel:

    ./vc guide 8b